In [1]:
import pandas as pd
import re
import json
import math
import os
from difflib import SequenceMatcher
from ollama import chat, ChatResponse
from tqdm import tqdm

In [2]:
# =========================
# Settings
# =========================
FILE_PATH = "./test/RACE_High_Combined.xlsx"

# 定義三個模型變數 (可隨時修改)
M1 = "gemma3:12b"
M2 = "qwen3:8b"
M3 = "deepseek-r1:8b"
M4 = "llama3.1:8b"
M5 = "mistral:7b"
M6 = "huihui_ai/internlm3-abliterated"
M7 = "ingu627/phi-4-mini-instruct-q4"
M8 = "glm4:9b"
M9 = "myaniu/minicpm4-8b:Q4_K_M"

# 第一階段：個別執行模型的清單
MODELS = [
    (M1, False), 
    (M2, False), 
    (M3, False),
    (M4, False),
    (M5, False),
    (M6, False),
    (M7, False),
    (M8, False),
    (M9, False)
]

CONTEXT_LEN = 1536
TEMPERATURE = 0.0

# 自動生成輸出檔名 (處理冒號問題)
def get_out_filename(model_name):
    clean_name = model_name.replace("/", "_").replace(":", "_")
    return f"result_{clean_name}.xlsx"

OUT_MODEL_FILE = {
    M1: get_out_filename(M1),
    M2: get_out_filename(M2),
    M3: get_out_filename(M3),
    M4: get_out_filename(M4),
    M5: get_out_filename(M5),
    M6: get_out_filename(M6),
    M7: get_out_filename(M7),
    M8: get_out_filename(M8),
    M9: get_out_filename(M9)
}

# Accuracy summary 輸出檔
OUT_SUMMARY_XLSX = "accuracy_summary.xlsx"
OUT_SUMMARY_CSV = "accuracy_summary.csv"

In [3]:
# =========================
# Ollama call
# =========================
def query_ollama(prompt, model, think=False) -> str:
    try:
        resp: ChatResponse = chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            options={"num_ctx": CONTEXT_LEN, "temperature": TEMPERATURE},
            think=think
        )
        return resp.message.content
    except Exception as e:
        print(f"[Ollama Error] model={model} err={e}")
        return ""

In [4]:
# =========================
# Utility: safe cast / NaN handling
# =========================
def is_nan(x) -> bool:
    try:
        return x is not None and isinstance(x, float) and math.isnan(x)
    except Exception:
        return False

def to_int_1_4(x):
    if x is None or pd.isna(x):
        return None
    try:
        xi = int(x)
    except Exception:
        try:
            s = str(x).strip()
            if s in ["1", "2", "3", "4"]:
                return int(s)
            return None
        except Exception:
            return None
    return xi if xi in (1, 2, 3, 4) else None

def to_conf_0_1(x):
    if x is None or pd.isna(x):
        return 0.0
    try:
        xf = float(x)
    except Exception:
        return 0.0
    if math.isnan(xf):
        return 0.0
    return max(0.0, min(1.0, xf))

In [5]:
# =========================
# Parsing helpers
# =========================
def _strip_think(text: str) -> str:
    if not text:
        return ""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

def _normalize(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"[\r\n\t]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _best_option_match(text_cleaned: str, options: dict) -> int | None:
    t = _normalize(text_cleaned)
    if not t:
        return None
    for k, opt in options.items():
        o = _normalize(opt)
        if o and (o in t or t in o):
            return int(k)
    best_k, best_score = None, 0.0
    for k, opt in options.items():
        o = _normalize(opt)
        if not o:
            continue
        score = SequenceMatcher(None, t, o).ratio()
        if score > best_score:
            best_k, best_score = k, score
    if best_k is not None and best_score >= 0.55:
        return int(best_k)
    return None
    
def parse_answer_and_confidence(raw_text: str, options: dict) -> tuple[int | None, float | None]:
    if not raw_text:
        return None, None
    text = _strip_think(raw_text)

    # A) JSON
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if m:
        try:
            obj = json.loads(m.group(0))
            ans = to_int_1_4(obj.get("answer", None))
            conf = obj.get("confidence", None)
            conf = None if conf is None else float(conf)
            if conf is not None:
                conf = max(0.0, min(1.0, conf))
            if ans is not None:
                return ans, conf
        except Exception:
            pass

    # B) Answer / Confidence
    m1 = re.search(r"Answer:\s*([1-4])", text, re.IGNORECASE)
    m2 = re.search(r"Confidence:\s*([0-9]*\.?[0-9]+)", text, re.IGNORECASE)
    ans = int(m1.group(1)) if m1 else None
    conf = float(m2.group(1)) if m2 else None
    if conf is not None:
        conf = max(0.0, min(1.0, conf))
    if ans is not None:
        return ans, conf

    # C) fallback: last digit 1-4
    nums = re.findall(r"\b([1-4])\b", text)
    if nums:
        return int(nums[-1]), conf

    # D) options match
    matched = _best_option_match(text, options)
    return matched, conf

In [6]:
# =========================
# Prompt builder
# =========================
def build_prompt(article: str, question: str, valid_options: dict) -> str:
    options_text = "\n".join([f"{k}. {v}" for k, v in valid_options.items()])
    valid_numbers = ", ".join([str(k) for k in valid_options.keys()])
    return f"""
Read the following article and answer the multiple-choice question.

Article:
{article}

Question:
{question}

Options:
{options_text}

Instruction:
1) Select the best option among ({valid_numbers}). If an option is missing/empty, do not select it.
2) Output ONLY a JSON object with:
   - "answer": an integer in [{valid_numbers}]
   - "confidence": a number in [0, 1] representing your probability the answer is correct.

Example:
{{"answer": 2, "confidence": 0.78}}
""".strip()

In [7]:
# =========================
# Step 1: Run one model over whole dataset
# =========================
def run_single_model(df: pd.DataFrame, model_name: str, think: bool) -> tuple[str, int, int, float]:
    out_file = OUT_MODEL_FILE.get(model_name, f"result_{model_name.replace(':', '_')}.xlsx")
    
    # 【修改1】如果檔案已存在，直接讀取不重跑
    if os.path.exists(out_file):
        print(f"\n[Info] {model_name} 的輸出檔已存在 ({out_file})，直接讀取結果...")
        res_df = pd.read_excel(out_file)
        correct = int(res_df["Correct"].sum())
        total = len(res_df)
        acc = (correct / total * 100) if total else 0.0
        print(f"{model_name} Accuracy: {correct}/{total} = {acc:.2f}%")
        return out_file, correct, total, acc

    print(f"\n====== Running {model_name} (think={think}) ======")
    correct = 0
    results = []
    total = len(df)

    for _, row in tqdm(df.iterrows(), total=total, desc=f"Predicting[{model_name}]"):
        options_all = {1: row["Option1"], 2: row["Option2"], 3: row["Option3"], 4: row["Option4"]}
        valid_options = {k: v for k, v in options_all.items() if v and str(v).lower() != "nan"}

        prompt = build_prompt(row["Article"], row["Question"], valid_options)
        raw = query_ollama(prompt, model=model_name, think=think)

        pred, conf = parse_answer_and_confidence(raw, valid_options)
        pred = to_int_1_4(pred)

        actual = to_int_1_4(row["Answer"])
        is_correct = (actual is not None and pred is not None and pred == actual)
        if is_correct:
            correct += 1

        results.append({
            "ID": row["ID"], "Actual": actual, "Predicted": pred,
            "Confidence": conf, "Correct": is_correct, "Model_Response": raw
        })

    acc = (correct / total * 100) if total else 0.0
    print(f"{model_name} Accuracy: {correct}/{total} = {acc:.2f}%")
    pd.DataFrame(results).to_excel(out_file, index=False)
    print(f"[Saved] {out_file}")
    return out_file, correct, total, acc

In [8]:
# =========================
# Entry
# =========================
print(f"Loading dataset: {FILE_PATH}")
df = pd.read_excel(FILE_PATH)

required_columns = ["ID", "Article", "Question", "Option1", "Option2", "Option3", "Option4", "Answer"]
if not all(c in df.columns for c in required_columns):
    raise ValueError(f"Excel 缺少必要欄位，需要: {required_columns}")

Loading dataset: ./test/RACE_High_Combined.xlsx


In [9]:
outputs = {}
summary_rows = []
model_accuracies = {}

# 1) 各模型跑完整 dataset (會自動偵測檔案略過)
for model_name, think in MODELS:
    out_file, correct, total, acc = run_single_model(df, model_name, think)
    outputs[model_name] = out_file
    model_accuracies[model_name] = acc
    summary_rows.append({
        "Method": model_name,
        "Correct": correct,
        "Total": total,
        "Accuracy(%)": round(acc, 4)
    })

# 找出準確率最高的模型
best_model_name = max(model_accuracies, key=model_accuracies.get)
print(f"\n[Info] 整體準確率最高的模型為: {best_model_name} (Accuracy: {model_accuracies[best_model_name]:.2f}%)")


[Info] gemma3:12b 的輸出檔已存在 (result_gemma3_12b.xlsx)，直接讀取結果...
gemma3:12b Accuracy: 2984/3498 = 85.31%

[Info] qwen3:8b 的輸出檔已存在 (result_qwen3_8b.xlsx)，直接讀取結果...
qwen3:8b Accuracy: 3040/3498 = 86.91%

[Info] deepseek-r1:8b 的輸出檔已存在 (result_deepseek-r1_8b.xlsx)，直接讀取結果...
deepseek-r1:8b Accuracy: 2962/3498 = 84.68%

====== Running llama3.1:8b (think=False) ======


Predicting[llama3.1:8b]: 100%|█████████████████████████████████████████████████████| 3498/3498 [58:45<00:00,  1.01s/it]


llama3.1:8b Accuracy: 2889/3498 = 82.59%
[Saved] result_llama3.1_8b.xlsx

====== Running mistral:7b (think=False) ======


Predicting[mistral:7b]: 100%|██████████████████████████████████████████████████████| 3498/3498 [14:27<00:00,  4.03it/s]


mistral:7b Accuracy: 2627/3498 = 75.10%
[Saved] result_mistral_7b.xlsx

====== Running huihui_ai/internlm3-abliterated (think=False) ======


...

ingu627/phi-4-mini-instruct-q4 Accuracy: 2725/3498 = 77.90%
[Saved] result_ingu627_phi-4-mini-instruct-q4.xlsx

====== Running glm4:9b (think=False) ======


Predicting[glm4:9b]: 100%|█████████████████████████████████████████████████████████| 3498/3498 [27:11<00:00,  2.14it/s]


glm4:9b Accuracy: 3080/3498 = 88.05%
[Saved] result_glm4_9b.xlsx

====== Running myaniu/minicpm4-8b:Q4_K_M (think=False) ======


Predicting[myaniu/minicpm4-8b:Q4_K_M]: 100%|███████████████████████████████████████| 3498/3498 [19:15<00:00,  3.03it/s]


myaniu/minicpm4-8b:Q4_K_M Accuracy: 2940/3498 = 84.05%
[Saved] result_myaniu_minicpm4-8b_Q4_K_M.xlsx

[Info] 整體準確率最高的模型為: huihui_ai/internlm3-abliterated (Accuracy: 88.05%)


In [10]:
summary_df = pd.DataFrame(summary_rows).sort_values(by="Accuracy(%)", ascending=False)
summary_df.to_excel(OUT_SUMMARY_XLSX, index=False)
summary_df.to_csv(OUT_SUMMARY_CSV, index=False, encoding="utf-8-sig")

print(f"\n[Saved] {OUT_SUMMARY_XLSX}")
print(f"[Saved] {OUT_SUMMARY_CSV}")
print("\n===== Accuracy Summary =====")
print(summary_df.to_string(index=False))


[Saved] accuracy_summary.xlsx
[Saved] accuracy_summary.csv

===== Accuracy Summary =====
                         Method  Correct  Total  Accuracy(%)
huihui_ai/internlm3-abliterated     3080   3498      88.0503
                        glm4:9b     3080   3498      88.0503
                       qwen3:8b     3040   3498      86.9068
                     gemma3:12b     2984   3498      85.3059
                 deepseek-r1:8b     2962   3498      84.6770
      myaniu/minicpm4-8b:Q4_K_M     2940   3498      84.0480
                    llama3.1:8b     2889   3498      82.5901
 ingu627/phi-4-mini-instruct-q4     2725   3498      77.9017
                     mistral:7b     2627   3498      75.1001
